# tirtl tutorial: TIRTL-seq pairing and analysis in Python (Rust backend)

This notebook shows how to use the `tirtl` package on the public TIRTL-seq data of the paper
(Zenodo [10.5281/zenodo.14010377](https://doi.org/10.5281/zenodo.14010377), experiment 3):

* **ID03**: CD8 T cells of a COVID patient, time point 2. We use the left half of the 384-well plate (192 wells).
* **ID02**: the same donor at time point 1. It is only used to compare the two time points.
* **10x exp3 tp2**: single-cell TCR-seq of the same sample, used as an independent check of the pairs.

`tirtl` reproduces the original R code. You get the same pairs and tables as the TIRTL pipeline and as TIRTLtools
(`run_pairing`, data processing, diversity), with the heavy steps running in Rust.

## 0. Installation and data

```bash
pip install -e .            # in the repository; needs a Rust toolchain (https://rustup.rs)
pip install cupy-cuda12x    # optional, NVIDIA GPU
pip install mlx             # optional, Apple Silicon GPU
```

Download and unpack the data (about 1.5 GB):
```bash
mkdir -p data/zenodo && cd data/zenodo
curl -L -o exp3_clones.tar.gz https://zenodo.org/api/records/14010377/files/exp3_clones.tar.gz/content
tar xzf exp3_clones.tar.gz && cd exp3_clones && for f in *.tar.gz; do tar xzf $f; done && cd ..
curl -L -o 10x.tar.gz https://zenodo.org/api/records/14010377/files/10x_scTCR_seq.tar.gz/content
mkdir -p 10x && tar xzf 10x.tar.gz -C 10x
```

In [ ]:
import os, time
from pathlib import Path
import pandas as pd
import tirtl

DATA = Path(os.environ.get("TIRTL_DATA", "data/zenodo"))
ID02 = DATA / "exp3_clones" / "TCR_clones_ID02"
ID03 = DATA / "exp3_clones" / "TCR_clones_ID03"
TENX = DATA / "10x" / "10x_scTCR_seq" / "exp3_tp2" / "filtered_contig_annotations.csv"
OUT = Path("tutorial_output"); OUT.mkdir(exist_ok=True)

print("tirtl", tirtl.__version__, "| Rust kernels:", tirtl.have_rust())
print(len(os.listdir(ID03)), "MiXCR files in", ID03.name)

tirtl 0.2.0 | Rust kernels: True
768 MiXCR files in TCR_clones_ID03


## 1. Pairing one plate with the original TIRTL function

`run_single_point_analysis_sub_gpu` takes the same arguments as the R function and writes the same
`<prefix>_TIRTLoutput.tsv`. Well ids are read from the file names (`well_pos`). `wellset1` selects
the wells; here these are the first 12 columns of the plate.

In [4]:
full_plate = tirtl.get_well_subset(range(1, 17), range(1, 25))
t = time.time()
pairs = tirtl.run_single_point_analysis_sub_gpu(str(ID03), prefix=str(OUT / "ID03"), wellset1=full_plate)
print(f"{time.time() - t:.1f} s")
pairs["method"].value_counts() 

start
2026-11-24T16:07:26 | rust kernels: True | backend: numpy
clonesets loaded
['MVP113_TIRTLseq_A1_S72_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A10_S216_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A11_S232_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A12_S248_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A13_S264_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A14_S280_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A15_S296_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A16_S312_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A17_S328_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A18_S344_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A19_S360_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A2_S88_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A20_S376_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A21_S392_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A22_S408_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A23_S424_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A24_S440_TCRa_ID03_clones_TRA_tsv', 'MVP113_TIRTLseq_A3_S

method
madhype    3424598
tshell       29143
Name: count, dtype: int64

In [6]:
pairs[["cdr3a", "va", "ja", "cdr3b", "vb", "jb", "method", "wij", "wa", "wb", "score", "r", "pval_adj"]].head()

,cdr3a,va,ja,cdr3b,vb,jb,method,wij,wa,wb,score,r,pval_adj
0,CAVLRGQKLLF,TRAV12-2,TRAJ16,CSSQEGGYGYTF,TRBV29-1,TRBJ1-2,madhype,199.0,228,211,53.077140,NaN,NaN
1,CAVLRGQKLLF,TRAV12-2,TRAJ16,CASSLLAEPFYEQYF,TRBV7-6,TRBJ2-7,madhype,195.0,228,202,56.392312,NaN,NaN
2,CAVLRGQKLLF,TRAV12-2,TRAJ16,CASYSSSYEQYF,TRBV27,TRBJ2-7,madhype,196.0,228,206,53.255427,NaN,NaN
3,CAVLRGQKLLF,TRAV12-2,TRAJ16,CASAVSSYEQYF,TRBV27,TRBJ2-7,madhype,191.0,228,197,54.954303,NaN,NaN
4,CAVLRGQKLLF,TRAV12-2,TRAJ16,CSSGGNTEAFF,TRBV29-1,TRBJ1-1,madhype,192.0,228,195,60.228393,NaN,NaN


**CPU / GPU.** By default (`engine="auto"`), MAD-HYPE and T-SHELL run in a sparse Rust kernel on the CPU.
It only compares chains that share a well, which is the fastest option for these data.

`engine="dense"` computes the correlations as float32 matrix products on `backend="numpy"`, `"cupy"` (NVIDIA)
or `"mlx"` (Apple Silicon), then re-scores the candidates exactly in float64. Every backend gives the same pairs:
```python
tirtl.run_single_point_analysis_sub_gpu(str(ID03), prefix="ID03_gpu", wellset1=half_plate,
                                        engine="dense", backend="cupy")
```

## 2. TIRTLtools `run_pairing` (one or several plates)

The TIRTLtools version adds well QC per chain and automatic T-SHELL settings. It also writes the pseudobulk
files that `load_tirtlseq` needs. To pair one sample spread over several plates, pass a list of folders
(and one well list per plate):
```python
tirtl.run_pairing(["plate1/", "plate2/"], "results/", "sample1", wellset=[wells_plate1, wells_plate2])
```
Here we pair the two time points (one plate each):

In [7]:
t = time.time()
for name, folder in (("ID02", ID02), ("ID03", ID03)):
    res = tirtl.run_pairing(str(folder), OUT / "samples", name, wellset=full_plate, verbose=False)
    print(name, res["method"].value_counts().to_dict())
print(f"{time.time() - t:.1f} s")
sorted(os.listdir(OUT / "samples"))

ID02 {'madhype': 1434417, 'tshell': 31212}
ID03 {'madhype': 3424598, 'tshell': 29143}
27.8 s


['ID02_TIRTLoutput.tsv',
 'ID02_pseudobulk_TRA.tsv',
 'ID02_pseudobulk_TRB.tsv',
 'ID03_TIRTLoutput.tsv',
 'ID03_pseudobulk_TRA.tsv',
 'ID03_pseudobulk_TRB.tsv']

## 3. Loading, cleaning and summarising samples

`load_tirtlseq` returns a `TIRTLseqDataSet`:
* `.meta` has one row per sample;
* `.data[sample]` holds four tables:
  * `alpha` and `beta`: pseudobulk single chains, with pairing annotations;
  * `paired`: one row per pair and method;
  * `paired_alt`: one row per unique pair.

In [8]:
ds = tirtl.load_tirtlseq(str(OUT / "samples"), verbose=False)
print(ds)
tirtl.summarize_data(ds)

TIRTLseqDataSet with 2 samples: ID02, ID03


,sample_id,n_alpha_clones,n_beta_clones,n_clone_pairs,n_reads_alpha,n_reads_beta
0,ID02,2836700,2780188,1436040,144220925,203003003
1,ID03,2888110,2668768,3426315,142696504,167784300


In [9]:
tirtl.get_pair_stats(ds, verbose=False).pivot_table(index=["sample_id", "chain"], columns="category",
                                                    values="Freq", fill_value=0, observed=True)

category          MAD-HYPE only  T-SHELL only     both    neither
sample_id chain                                                  
ID02      alpha         30745.0        1042.0  27800.0  2777113.0
          beta          32892.0         959.0  23328.0  2723009.0
          paired      1404828.0        1623.0  29589.0        0.0
ID03      alpha         34808.0        1216.0  25440.0  2826646.0
          beta          38153.0        1089.0  20957.0  2608569.0
          paired      3397172.0        1717.0  27426.0        0.0

In [10]:
# keep at most 2 alphas per beta and 1 beta per alpha (most frequent partners first)
clean = tirtl.clean_pairs(ds, verbose=False)
p = clean.data["ID03"]["paired"]
print(len(ds.data["ID03"]["paired"]), "->", len(p), "pairs")
p = tirtl.filter_mait(tirtl.filter_nonfunctional_TCRs(p, verbose=False))
p[["cdr3a", "va", "cdr3b", "vb", "alpha_readFraction", "beta_readFraction", "method"]].head()

3453741 -> 47635 pairs
Removed 331 MAIT TCRs (0.97%) from a total of 34,134 TCRs.


,cdr3a,va,cdr3b,vb,alpha_readFraction,beta_readFraction,method
0,TGGADGLTF,TRAV8-2,CASDRDRASYEQYF,TRBV28,5.683391e-06,6.955359e-06,madhype
1,TRRRNAGNMLTF,TRAV27,CASSPGTGGQPQHF,TRBV18,2.817168e-06,3.236298e-06,madhype
2,TGNQFYF,TRAV24,CASRVSRDIGSPLHF,TRBV19,8.332370e-06,1.490008e-07,madhype
3,TGRDDKIIF,TRAV9-2,CASIPTVQHIEKLFF,TRBV6-6,1.962206e-07,1.668809e-07,madhype
4,TNGLRFIQGAQKLVF,TRAV13-2,CSIGGGSSNQPQHF,TRBV20-1,9.180323e-07,1.537689e-06,madhype


## 4. Diversity

`calculate_diversity` computes the 14 metrics of `get_all_div_metrics()`.

In [11]:
dv = tirtl.calculate_diversity(ds, chain="beta", verbose=False)
pd.DataFrame({s: {m: r["diversity"][m] for m in ("richness", "shannon", "simpson", "gini", "top10fraction")}
              for s, r in dv["result"].items()})

,ID02,ID03
richness,2.780188e+06,2.668768e+06
shannon,1.263429e+01,1.214311e+01
simpson,2.034372e-04,5.988665e-04
gini,8.296061e-01,8.425735e-01
top10fraction,3.454366e-02,6.256061e-02


In [12]:
dv["result"]["ID03"]["diversity"]["dXX"]   # number of clones needed to reach XX % of the repertoire

,n_clones,percent_required,percent_observed
0,36,10,10.063995
1,556,20,20.003431
2,4748,30,30.000827
3,21984,40,40.000284
4,64458,50,50.000116
5,137444,60,60.000001
6,244216,70,70.000059
7,393571,80,80.000021
8,615581,90,90.000032


## 5. Comparing two time points

The TIRTLtools approach compares the pseudobulk frequencies (with SEM smoothing) and returns the paired TCRs
of the expanded and contracted clones:

In [13]:
ex = tirtl.get_expanded_clones(ds.data["ID02"], ds.data["ID03"], chain="beta")
print(len(ex["expanded"]["single_chain"]), "expanded and", len(ex["contracted"]["single_chain"]), "contracted beta clones")
ex["expanded"]["paired"][["sign", "log2FC", "cdr3a", "va", "cdr3b", "vb", "readFraction.x", "readFraction.y"]].head()

1207 expanded and 141 contracted beta clones


,sign,log2FC,cdr3a,va,cdr3b,vb,readFraction.x,readFraction.y
0,up,13.699253,CAVLRGQKLLF,TRAV12-2,CSSQEGGYGYTF,TRBV29-1,1.034467e-07,0.014676
1,up,13.699253,CAGLSDGQKLLF,TRAV12-2,CSSQEGGYGYTF,TRBV29-1,1.034467e-07,0.014676
2,up,13.699253,CAYRSFPYNNNDMRF,TRAV38-2/DV8,CSSQEGGYGYTF,TRBV29-1,1.034467e-07,0.014676
3,up,13.699253,CVVSYSGGYQKVTF,TRAV8-2,CSSQEGGYGYTF,TRBV29-1,1.034467e-07,0.014676
4,up,13.699253,CAVPWANDMRF,TRAV38-2/DV8,CSSQEGGYGYTF,TRBV29-1,1.034467e-07,0.014676


The original TIRTL function works directly on the MiXCR folders (clones keyed by `nSeqCDR3`). It uses its
own well QC (wells with more than 0.75 × the mean number of clones per file). We run it on the full plates,
because with this threshold no well of the left half of ID02 passes QC.

In [14]:
t = time.time()
lg = tirtl.run_longitudinal_analysis_sub(ID02, ID03)   # full plates (default wellsets)
print(f"{time.time() - t:.1f} s")
lg["beta"]["sign"].value_counts()

12.6 s


sign
stable    97452
up          539
down         30
Name: count, dtype: int64

## 6. Comparison with 10x single-cell TCR-seq

In [15]:
tenx = tirtl.process_10x(TENX)
print(len(tenx["complete"]), "10x clonotypes with both chains")
# among TIRTL pairs whose beta chain is in the 10x data, how many have the same alpha?
tirtl.concordance(pairs, tenx["complete"])

10586 10x clonotypes with both chains


FALSE    1284962
TRUE        3320
Name: N, dtype: int64

## 7. Speed

The chart below compares running times on these public data:
* the original R code, and TIRTLtools 0.2.9 (C++ backend);
* tirtl (Python + Rust) on the same machine.

The results are the same as R. `docs/make_benchmark_plot.py` regenerates the chart.

![benchmark](docs/benchmark_public.png)